In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 21.1 - Overview, paths, and fixed mapped-window refinement design
# Purpose:
# Refine the four mapped broad groups supported by Notebook 20:
# W01, W04, W07, and W10.
#
# Each parent window is divided into 10 equal genomic subwindows using the
# fixed MG1655 reference coordinates already established in Notebook 19.
#
# For every subwindow:
# 1. remove that subwindow from the full Notebook 18 unitig kernel;
# 2. refit the same REML model to continuous log2 ceftazidime MIC;
# 3. measure the fall from the full variance fraction;
# 4. compare that fall with matched random removals drawn ONLY from the
#    same parent window.
#
# Random removals are matched exactly for:
# - number of unitigs;
# - unitig presence-count distribution across the 176 pathogens;
# - therefore the kernel denominator contribution.
#
# There are 10 subwindows per parent and 250 matched random partitions.
# BH correction is applied separately within each pre-selected parent window.
#
# UNMAPPED and MULTIMAPPED are not refined here because they require a
# different phenotype-independent grouping strategy.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from scipy import sparse, optimize
from IPython.display import display
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / "03_Notebooks" / "04_Genome_Comparison"
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"

UNITIG_DIR = PROJECT_ROOT / "04_Intermediate" / "10_Whole_Chromosome_Unitigs"
UNITIG_MATRIX = UNITIG_DIR / "10_variable_unitig_matrix_176xM.npz"
UNITIG_SAMPLES = UNITIG_DIR / "10_unitig_sample_order.csv"

NB18_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "18_Collective_Whole_Chromosome_Association"
)

NB18_K = NB18_DIR / "18_whole_sequence_unitig_similarity_matrix.npz"
NB18_SUMMARY = RESULTS_TABLE_DIR / "18_collective_unitig_association_summary.csv"
NB18_QC = RESULTS_TABLE_DIR / "18_collective_unitig_association_final_QC.csv"

NB19_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "19_Broad_Unitig_Ablation"
)

NB19_ASSIGNMENT = NB19_DIR / "19_unitig_reference_assignment.npz"
NB19_COMPONENTS = NB19_DIR / "19_group_kernel_components.npz"
NB19_GROUP_MANIFEST = RESULTS_TABLE_DIR / "19_broad_ablation_group_manifest.csv"
NB19_QC = RESULTS_TABLE_DIR / "19_broad_ablation_final_QC.csv"

NB20_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "20_Matched_Random_Ablation_Benchmark"
)

NB20_FINAL_RESULTS = RESULTS_TABLE_DIR / "20_matched_random_ablation_final_results.csv"
NB20_QC = RESULTS_TABLE_DIR / "20_matched_random_ablation_final_QC.csv"

NB21_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "21_Mapped_Priority_Window_Subwindow_Ablation"
)

NB21_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUBWINDOW_ASSIGNMENT = (
    NB21_DIR
    / "21_mapped_subwindow_assignment.npz"
)

SUBWINDOW_COMPONENTS = (
    NB21_DIR
    / "21_mapped_subwindow_kernel_components.npz"
)

PROGRESS_FILE = (
    NB21_DIR
    / "21_matched_random_subwindow_ablation_progress.csv.gz"
)

SMOKE_TEST_FILE = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_ablation_smoke_test.csv"
)

SUBWINDOW_MANIFEST = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_manifest.csv"
)

OBSERVED_RESULTS = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_observed_results.csv"
)

FINAL_RESULTS = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_matched_random_results.csv"
)

FINAL_QC = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_final_QC.csv"
)

COMPLETION_FILE = (
    NB21_DIR
    / "21_MAPPED_SUBWINDOW_ABLATION_COMPLETE.json"
)

EXPECTED_PATHOGENS = 176
EXPECTED_UNITIGS = 1_287_844
EXPECTED_BROAD_GROUPS = 12

PRIORITY_PARENTS = [
    "W01",
    "W04",
    "W07",
    "W10",
]

N_SUBWINDOWS_PER_PARENT = 10
EXPECTED_SUBWINDOWS = (
    len(PRIORITY_PARENTS)
    * N_SUBWINDOWS_PER_PARENT
)

N_RANDOM_PARTITIONS_PER_PARENT = 250
RANDOM_SEED_BASE = 2026092100

EXPECTED_FULL_VARIANCE_FRACTION = 0.538815

for path in [
    PROJECT_ROOT,
    NOTEBOOK_DIR,
    RESULTS_TABLE_DIR,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    NB18_K,
    NB18_SUMMARY,
    NB18_QC,
    NB19_ASSIGNMENT,
    NB19_COMPONENTS,
    NB19_GROUP_MANIFEST,
    NB19_QC,
    NB20_FINAL_RESULTS,
    NB20_QC,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Notebook 21 - Mapped Priority Window Subwindow Ablation")
print("Priority mapped parents:", ", ".join(PRIORITY_PARENTS))
print("Subwindows per parent:", N_SUBWINDOWS_PER_PARENT)
print("Total mapped subwindows:", EXPECTED_SUBWINDOWS)
print(
    "Matched random partitions per parent:",
    N_RANDOM_PARTITIONS_PER_PARENT,
)
print(
    "Multiple-testing correction:",
    "BH separately across 10 subwindows within each parent",
)
print("UNMAPPED and MULTIMAPPED are deferred to a separate refinement.")
print(
    "\nTransition: Cell 21.2 will verify Notebooks 18-20 and load "
    "the exact kernel, phenotype, and reference assignments."
)


In [ ]:
#@title Cell 21.2 - Verify Notebooks 18-20 and load authoritative inputs
# Purpose:
# Confirm that the four mapped parent windows were supported by Notebook 20,
# recover the full Notebook 18 kernel decomposition, and load the exact
# MG1655 midpoint assignment from Notebook 19.

nb18_qc = pd.read_csv(
    NB18_QC
)

nb19_qc = pd.read_csv(
    NB19_QC
)

nb20_qc = pd.read_csv(
    NB20_QC
)

assert len(nb18_qc) == 1
assert len(nb19_qc) == 1
assert len(nb20_qc) == 1

assert bool(
    nb18_qc.loc[
        0,
        "final_QC_pass",
    ]
)

assert bool(
    nb19_qc.loc[
        0,
        "final_QC_pass",
    ]
)

assert bool(
    nb20_qc.loc[
        0,
        "final_QC_pass",
    ]
)

nb18_summary = pd.read_csv(
    NB18_SUMMARY
)

baseline_variance_fraction = float(
    nb18_summary.loc[
        0,
        "whole_sequence_unitig_variance_fraction",
    ]
)

assert abs(
    baseline_variance_fraction
    - EXPECTED_FULL_VARIANCE_FRACTION
) < 0.001

samples = (
    pd.read_csv(
        UNITIG_SAMPLES
    )
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    samples
) == EXPECTED_PATHOGENS

assert np.array_equal(
    samples[
        "sample_index"
    ].to_numpy(
        dtype=int
    ),
    np.arange(
        EXPECTED_PATHOGENS
    ),
)

y = samples[
    "log2_mic"
].to_numpy(
    dtype=float
)

assert y.shape == (
    EXPECTED_PATHOGENS,
)

assert np.isfinite(
    y
).all()

with np.load(
    NB18_K
) as archive:
    K_full = np.asarray(
        archive[
            "K_unitig"
        ],
        dtype=float,
    )

K_full = (
    K_full
    + K_full.T
) / 2.0

assert K_full.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

group_manifest = (
    pd.read_csv(
        NB19_GROUP_MANIFEST
    )
    .sort_values(
        "group_code"
    )
    .reset_index(
        drop=True
    )
)

assert len(
    group_manifest
) == EXPECTED_BROAD_GROUPS

with np.load(
    NB19_ASSIGNMENT
) as archive:
    broad_group_code = np.asarray(
        archive[
            "group_code"
        ],
        dtype=np.uint8,
    )

    alignment_count = np.asarray(
        archive[
            "alignment_count"
        ],
        dtype=np.uint8,
    )

    reference_midpoint = np.asarray(
        archive[
            "reference_midpoint"
        ],
        dtype=np.int32,
    )

    broad_window_edges = np.asarray(
        archive[
            "window_edges"
        ],
        dtype=np.int64,
    )

assert broad_group_code.shape == (
    EXPECTED_UNITIGS,
)

assert alignment_count.shape == (
    EXPECTED_UNITIGS,
)

assert reference_midpoint.shape == (
    EXPECTED_UNITIGS,
)

with np.load(
    NB19_COMPONENTS
) as archive:
    broad_group_numerators = np.asarray(
        archive[
            "group_numerators"
        ],
        dtype=float,
    )

    broad_group_denominators = np.asarray(
        archive[
            "group_denominators"
        ],
        dtype=float,
    )

    broad_group_unitig_counts = np.asarray(
        archive[
            "group_unitig_counts"
        ],
        dtype=np.int64,
    )

assert broad_group_numerators.shape == (
    EXPECTED_BROAD_GROUPS,
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

total_numerator = np.sum(
    broad_group_numerators,
    axis=0,
)

total_denominator = float(
    np.sum(
        broad_group_denominators
    )
)

K_reconstructed = (
    total_numerator
    / total_denominator
)

K_reconstructed = (
    K_reconstructed
    + K_reconstructed.T
) / 2.0

full_reconstruction_error = float(
    np.max(
        np.abs(
            K_reconstructed
            - K_full
        )
    )
)

assert full_reconstruction_error < 1e-10

nb20_results = pd.read_csv(
    NB20_FINAL_RESULTS
)

required_nb20_columns = {
    "group_name",
    "larger_drop_than_matched_random_after_BH",
}

assert required_nb20_columns.issubset(
    nb20_results.columns
)

priority_check = (
    nb20_results
    .loc[
        nb20_results[
            "group_name"
        ].isin(
            PRIORITY_PARENTS
        ),
        [
            "group_name",
            "observed_drop_after_removal",
            "empirical_p_value",
            "Benjamini_Hochberg_q_value",
            "larger_drop_than_matched_random_after_BH",
        ],
    ]
    .copy()
)

assert set(
    priority_check[
        "group_name"
    ]
) == set(
    PRIORITY_PARENTS
)

assert priority_check[
    "larger_drop_than_matched_random_after_BH"
].astype(
    bool
).all()

parent_code_by_name = {
    str(
        row[
            "group_name"
        ]
    ): int(
        row[
            "group_code"
        ]
    )
    for _, row in group_manifest.iterrows()
}

for parent_name in PRIORITY_PARENTS:
    assert parent_name in parent_code_by_name

print("Notebook 18 QC: PASS")
print("Notebook 19 QC: PASS")
print("Notebook 20 QC: PASS")
print("Full variance fraction:", baseline_variance_fraction)
print("Full K reconstruction error:", full_reconstruction_error)
print("\nMapped parent windows confirmed for refinement:")

display(
    priority_check
    .sort_values(
        "group_name"
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nCell 21.2 complete."
)

print(
    "Transition: Cell 21.3 will divide W01, W04, W07, and W10 "
    "into 10 fixed equal genomic subwindows each."
)


In [ ]:
#@title Cell 21.3 - Divide the four mapped parents into 40 fixed subwindows
# Purpose:
# Split each selected parent window into 10 equal MG1655 coordinate intervals.
# Assignment uses only the Notebook 19 reference midpoint and is independent
# of MIC.
#
# Every unitig in each selected parent must enter exactly one subwindow.

subwindow_code = np.full(
    EXPECTED_UNITIGS,
    -1,
    dtype=np.int16,
)

manifest_rows = []

global_subwindow_code = 0

for parent_order, parent_name in enumerate(
    PRIORITY_PARENTS
):
    parent_row = group_manifest.loc[
        group_manifest[
            "group_name"
        ]
        == parent_name
    ].iloc[0]

    parent_group_code = int(
        parent_row[
            "group_code"
        ]
    )

    parent_start = int(
        parent_row[
            "reference_start_0_based"
        ]
    )

    parent_end = int(
        parent_row[
            "reference_end_0_based_exclusive"
        ]
    )

    parent_mask = (
        broad_group_code
        == parent_group_code
    )

    parent_indices = np.flatnonzero(
        parent_mask
    )

    parent_midpoints = reference_midpoint[
        parent_indices
    ]

    assert len(
        parent_indices
    ) == int(
        parent_row[
            "n_unitigs"
        ]
    )

    assert np.all(
        alignment_count[
            parent_indices
        ]
        == 1
    )

    assert np.all(
        parent_midpoints
        >= parent_start
    )

    assert np.all(
        parent_midpoints
        < parent_end
    )

    parent_width = (
        parent_end
        - parent_start
    )

    sub_edges = (
        parent_start
        + (
            parent_width
            * np.arange(
                N_SUBWINDOWS_PER_PARENT + 1,
                dtype=np.int64,
            )
        )
        // N_SUBWINDOWS_PER_PARENT
    )

    assert sub_edges[0] == parent_start
    assert sub_edges[-1] == parent_end
    assert np.all(
        np.diff(
            sub_edges
        ) > 0
    )

    local_codes = np.searchsorted(
        sub_edges[
            1:
        ],
        parent_midpoints,
        side="right",
    ).astype(
        np.int16
    )

    assert local_codes.min() >= 0
    assert local_codes.max() < N_SUBWINDOWS_PER_PARENT

    for local_code in range(
        N_SUBWINDOWS_PER_PARENT
    ):
        current_global_code = global_subwindow_code

        current_mask_in_parent = (
            local_codes
            == local_code
        )

        current_indices = parent_indices[
            current_mask_in_parent
        ]

        assert len(
            current_indices
        ) > 0

        subwindow_code[
            current_indices
        ] = current_global_code

        subwindow_name = (
            f"{parent_name}_S{local_code + 1:02d}"
        )

        start_0 = int(
            sub_edges[
                local_code
            ]
        )

        end_0 = int(
            sub_edges[
                local_code + 1
            ]
        )

        manifest_rows.append(
            {
                "global_subwindow_code": current_global_code,
                "parent_order": parent_order,
                "parent_group_code": parent_group_code,
                "parent_group_name": parent_name,
                "local_subwindow_code": local_code,
                "subwindow_name": subwindow_name,
                "reference_start_0_based": start_0,
                "reference_end_0_based_exclusive": end_0,
                "reference_width_bp": (
                    end_0
                    - start_0
                ),
                "n_unitigs": len(
                    current_indices
                ),
                "fraction_of_parent_unitigs": (
                    len(
                        current_indices
                    )
                    / len(
                        parent_indices
                    )
                ),
            }
        )

        global_subwindow_code += 1

assert global_subwindow_code == EXPECTED_SUBWINDOWS

subwindow_manifest = pd.DataFrame(
    manifest_rows
)

selected_parent_mask = np.isin(
    broad_group_code,
    [
        parent_code_by_name[
            name
        ]
        for name in PRIORITY_PARENTS
    ],
)

assert np.all(
    subwindow_code[
        selected_parent_mask
    ]
    >= 0
)

assert np.all(
    subwindow_code[
        ~selected_parent_mask
    ]
    == -1
)

for parent_name in PRIORITY_PARENTS:
    parent_group_code = parent_code_by_name[
        parent_name
    ]

    observed_parent_count = int(
        np.sum(
            broad_group_code
            == parent_group_code
        )
    )

    subwindow_parent_count = int(
        subwindow_manifest.loc[
            subwindow_manifest[
                "parent_group_name"
            ]
            == parent_name,
            "n_unitigs",
        ].sum()
    )

    assert (
        observed_parent_count
        == subwindow_parent_count
    )

np.savez_compressed(
    SUBWINDOW_ASSIGNMENT,
    subwindow_code=subwindow_code,
    priority_parent_names=np.asarray(
        PRIORITY_PARENTS,
        dtype="U3",
    ),
)

subwindow_manifest.to_csv(
    SUBWINDOW_MANIFEST,
    index=False,
)

print("Subwindow assignment: PASS")
print(
    "Selected mapped unitigs:",
    f"{int(selected_parent_mask.sum()):,}",
)

display(
    subwindow_manifest
)

print(
    "\nCell 21.3 complete."
)

print(
    "Transition: Cell 21.4 will decompose the 40 subwindows, verify "
    "exact reconstruction of each parent kernel, and fit observed ablations."
)


In [ ]:
#@title Cell 21.4 - Decompose subwindow kernels and fit observed leave-one-subwindow-out effects
# Purpose:
# Calculate the exact kernel contribution of every subwindow.
#
# The 10 subwindow components inside each parent must exactly reconstruct
# that parent's Notebook 19 kernel component.
#
# Then remove each subwindow from the full Notebook 18 kernel and measure
# the observed fall in REML variance fraction.

X = sparse.load_npz(
    UNITIG_MATRIX
).tocsc()

assert X.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_UNITIGS,
)

presence_count = np.asarray(
    X.sum(
        axis=0
    )
).ravel().astype(
    np.int16
)

assert presence_count.min() >= 1
assert presence_count.max() <= (
    EXPECTED_PATHOGENS - 1
)

def numerator_from_columns(columns):
    X_group = X[
        :,
        columns,
    ]

    count_group = presence_count[
        columns
    ].astype(
        np.float64
    )

    p_group = (
        count_group
        / EXPECTED_PATHOGENS
    )

    denominator_group = float(
        np.sum(
            p_group
            * (
                1.0
                - p_group
            )
        )
    )

    XX_group = (
        X_group.astype(
            np.int32
        )
        @ X_group.astype(
            np.int32
        ).T
    ).toarray().astype(
        np.float64
    )

    Xp_group = np.asarray(
        X_group
        @ p_group
    ).reshape(-1).astype(
        np.float64
    )

    p2_group = float(
        p_group
        @ p_group
    )

    numerator_group = (
        XX_group
        - Xp_group[:, None]
        - Xp_group[None, :]
        + p2_group
    )

    numerator_group = (
        numerator_group
        + numerator_group.T
    ) / 2.0

    return (
        numerator_group,
        denominator_group,
    )

def prepare_kernel(K):
    K = np.asarray(
        K,
        dtype=float,
    )

    K = (
        K
        + K.T
    ) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(
        K
    )

    minimum_eigenvalue = float(
        eigenvalues.min()
    )

    if minimum_eigenvalue < -1e-6:
        raise ValueError(
            "Kernel is not positive semidefinite: "
            f"minimum eigenvalue = {minimum_eigenvalue}"
        )

    eigenvalues = np.maximum(
        eigenvalues,
        0.0,
    )

    transformed_intercept = (
        eigenvectors.T
        @ np.ones(
            K.shape[0],
            dtype=float,
        )
    )

    return {
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "transformed_intercept": transformed_intercept,
    }

def fit_null_reml_prepared(y_input, prepared):
    y_input = np.asarray(
        y_input,
        dtype=float,
    ).reshape(-1)

    eigenvalues = prepared[
        "eigenvalues"
    ]

    eigenvectors = prepared[
        "eigenvectors"
    ]

    transformed_intercept = prepared[
        "transformed_intercept"
    ]

    transformed_y = (
        eigenvectors.T
        @ y_input
    )

    n = len(
        y_input
    )

    degrees_of_freedom = (
        n - 1
    )

    def evaluate_ratio(ratio):
        if ratio < 0:
            return None

        covariance_eigenvalues = (
            1.0
            + ratio
            * eigenvalues
        )

        if np.any(
            covariance_eigenvalues <= 0
        ):
            return None

        inverse_weights = (
            1.0
            / covariance_eigenvalues
        )

        information = float(
            np.sum(
                transformed_intercept
                * transformed_intercept
                * inverse_weights
            )
        )

        if information <= 0:
            return None

        beta_0 = float(
            np.sum(
                transformed_intercept
                * transformed_y
                * inverse_weights
            )
            / information
        )

        transformed_residual = (
            transformed_y
            - beta_0
            * transformed_intercept
        )

        residual_quadratic = float(
            np.sum(
                transformed_residual
                * transformed_residual
                * inverse_weights
            )
        )

        if residual_quadratic <= 0:
            return None

        sigma_e2 = (
            residual_quadratic
            / degrees_of_freedom
        )

        sigma_g2 = (
            ratio
            * sigma_e2
        )

        objective = 0.5 * (
            degrees_of_freedom
            * np.log(
                sigma_e2
            )
            + np.log(
                covariance_eigenvalues
            ).sum()
            + np.log(
                information
            )
        )

        return {
            "objective": float(
                objective
            ),
            "variance_fraction": float(
                sigma_g2
                / (
                    sigma_g2
                    + sigma_e2
                )
            ),
        }

    def objective_on_log_ratio(log_ratio):
        result = evaluate_ratio(
            np.exp(
                log_ratio
            )
        )

        if result is None:
            return np.inf

        return result[
            "objective"
        ]

    optimized = optimize.minimize_scalar(
        objective_on_log_ratio,
        bounds=(
            -12.0,
            12.0,
        ),
        method="bounded",
        options={
            "xatol": 1e-8,
            "maxiter": 500,
        },
    )

    candidates = []

    zero_result = evaluate_ratio(
        0.0
    )

    if zero_result is not None:
        candidates.append(
            zero_result
        )

    if optimized.success:
        optimized_result = evaluate_ratio(
            float(
                np.exp(
                    optimized.x
                )
            )
        )

        if optimized_result is not None:
            candidates.append(
                optimized_result
            )

    high_result = evaluate_ratio(
        float(
            np.exp(
                12.0
            )
        )
    )

    if high_result is not None:
        candidates.append(
            high_result
        )

    assert candidates

    return min(
        candidates,
        key=lambda item: item[
            "objective"
        ],
    )

subwindow_numerators = np.zeros(
    (
        EXPECTED_SUBWINDOWS,
        EXPECTED_PATHOGENS,
        EXPECTED_PATHOGENS,
    ),
    dtype=np.float64,
)

subwindow_denominators = np.zeros(
    EXPECTED_SUBWINDOWS,
    dtype=np.float64,
)

decomposition_start = time.time()

for subwindow_index in range(
    EXPECTED_SUBWINDOWS
):
    columns = np.flatnonzero(
        subwindow_code
        == subwindow_index
    )

    assert len(
        columns
    ) > 0

    (
        numerator_group,
        denominator_group,
    ) = numerator_from_columns(
        columns
    )

    subwindow_numerators[
        subwindow_index
    ] = numerator_group

    subwindow_denominators[
        subwindow_index
    ] = denominator_group

# Verify exact reconstruction of each parent component.
parent_reconstruction_rows = []

for parent_name in PRIORITY_PARENTS:
    parent_code = parent_code_by_name[
        parent_name
    ]

    sub_codes = (
        subwindow_manifest.loc[
            subwindow_manifest[
                "parent_group_name"
            ]
            == parent_name,
            "global_subwindow_code",
        ]
        .to_numpy(
            dtype=int
        )
    )

    reconstructed_parent_numerator = np.sum(
        subwindow_numerators[
            sub_codes
        ],
        axis=0,
    )

    reconstructed_parent_denominator = float(
        np.sum(
            subwindow_denominators[
                sub_codes
            ]
        )
    )

    numerator_error = float(
        np.max(
            np.abs(
                reconstructed_parent_numerator
                - broad_group_numerators[
                    parent_code
                ]
            )
        )
    )

    denominator_error = float(
        abs(
            reconstructed_parent_denominator
            - broad_group_denominators[
                parent_code
            ]
        )
    )

    assert numerator_error < 1e-8
    assert denominator_error < 1e-8

    parent_reconstruction_rows.append(
        {
            "parent_group_name": parent_name,
            "maximum_numerator_reconstruction_error": numerator_error,
            "denominator_reconstruction_error": denominator_error,
        }
    )

parent_reconstruction = pd.DataFrame(
    parent_reconstruction_rows
)

subwindow_manifest = subwindow_manifest.copy()

subwindow_manifest[
    "kernel_denominator_contribution"
] = subwindow_denominators

for parent_name in PRIORITY_PARENTS:
    parent_code = parent_code_by_name[
        parent_name
    ]

    mask = (
        subwindow_manifest[
            "parent_group_name"
        ]
        == parent_name
    )

    subwindow_manifest.loc[
        mask,
        "fraction_of_parent_kernel_denominator",
    ] = (
        subwindow_manifest.loc[
            mask,
            "kernel_denominator_contribution",
        ]
        / broad_group_denominators[
            parent_code
        ]
    )

subwindow_manifest.to_csv(
    SUBWINDOW_MANIFEST,
    index=False,
)

np.savez_compressed(
    SUBWINDOW_COMPONENTS,
    subwindow_numerators=subwindow_numerators,
    subwindow_denominators=subwindow_denominators,
)

observed_rows = []

for subwindow_index in range(
    EXPECTED_SUBWINDOWS
):
    denominator_group = float(
        subwindow_denominators[
            subwindow_index
        ]
    )

    leave_out_K = (
        total_numerator
        - subwindow_numerators[
            subwindow_index
        ]
    ) / (
        total_denominator
        - denominator_group
    )

    leave_out_K = (
        leave_out_K
        + leave_out_K.T
    ) / 2.0

    prepared = prepare_kernel(
        leave_out_K
    )

    fit = fit_null_reml_prepared(
        y,
        prepared,
    )

    leave_out_fraction = float(
        fit[
            "variance_fraction"
        ]
    )

    observed_drop = (
        baseline_variance_fraction
        - leave_out_fraction
    )

    manifest_row = subwindow_manifest.loc[
        subwindow_manifest[
            "global_subwindow_code"
        ]
        == subwindow_index
    ].iloc[0]

    observed_rows.append(
        {
            "global_subwindow_code": subwindow_index,
            "parent_group_name": str(
                manifest_row[
                    "parent_group_name"
                ]
            ),
            "local_subwindow_code": int(
                manifest_row[
                    "local_subwindow_code"
                ]
            ),
            "subwindow_name": str(
                manifest_row[
                    "subwindow_name"
                ]
            ),
            "reference_start_0_based": int(
                manifest_row[
                    "reference_start_0_based"
                ]
            ),
            "reference_end_0_based_exclusive": int(
                manifest_row[
                    "reference_end_0_based_exclusive"
                ]
            ),
            "n_unitigs": int(
                manifest_row[
                    "n_unitigs"
                ]
            ),
            "fraction_of_parent_kernel_denominator": float(
                manifest_row[
                    "fraction_of_parent_kernel_denominator"
                ]
            ),
            "leave_one_subwindow_out_variance_fraction": leave_out_fraction,
            "absolute_drop_after_removal": observed_drop,
        }
    )

observed_results = pd.DataFrame(
    observed_rows
)

observed_results.to_csv(
    OBSERVED_RESULTS,
    index=False,
)

print("Parent-kernel reconstruction: PASS")
display(
    parent_reconstruction
)

print(
    "\nObserved subwindow ablations, ranked within each parent:"
)

display(
    observed_results
    .sort_values(
        [
            "parent_group_name",
            "absolute_drop_after_removal",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nDecomposition and observed fitting elapsed:",
    f"{(time.time() - decomposition_start) / 60.0:.1f} minutes",
)

print(
    "\nCell 21.4 complete."
)

print(
    "Transition: Cell 21.5 will run one matched-random partition "
    "inside each parent window as a smoke test."
)


In [ ]:
#@title Cell 21.5 - Matched-random smoke test within each mapped parent
# Purpose:
# Verify the parent-specific matched-random design before the long benchmark.
#
# Within each parent window, random subwindow labels are shuffled only among
# unitigs with the same presence count across the 176 pathogens.
#
# Thus every random counterpart preserves exactly:
# - its observed number of unitigs;
# - its observed presence-count histogram;
# - its observed kernel denominator contribution.

parent_randomization_data = {}

for parent_order, parent_name in enumerate(
    PRIORITY_PARENTS
):
    parent_code = parent_code_by_name[
        parent_name
    ]

    parent_unitig_indices = np.flatnonzero(
        broad_group_code
        == parent_code
    )

    parent_subwindow_rows = (
        subwindow_manifest.loc[
            subwindow_manifest[
                "parent_group_name"
            ]
            == parent_name
        ]
        .sort_values(
            "local_subwindow_code"
        )
        .reset_index(
            drop=True
        )
    )

    assert len(
        parent_subwindow_rows
    ) == N_SUBWINDOWS_PER_PARENT

    global_codes = parent_subwindow_rows[
        "global_subwindow_code"
    ].to_numpy(
        dtype=int
    )

    global_to_local = {
        int(
            global_code
        ): int(
            local_code
        )
        for global_code, local_code in zip(
            global_codes,
            parent_subwindow_rows[
                "local_subwindow_code"
            ].to_numpy(
                dtype=int
            ),
        )
    }

    observed_local_code = np.asarray(
        [
            global_to_local[
                int(
                    global_code
                )
            ]
            for global_code in subwindow_code[
                parent_unitig_indices
            ]
        ],
        dtype=np.int8,
    )

    parent_presence_count = presence_count[
        parent_unitig_indices
    ]

    stratum_positions = {}
    stratum_local_counts = {}

    for count in np.unique(
        parent_presence_count
    ):
        positions = np.flatnonzero(
            parent_presence_count
            == count
        )

        local_counts = np.bincount(
            observed_local_code[
                positions
            ],
            minlength=N_SUBWINDOWS_PER_PARENT,
        ).astype(
            np.int64
        )

        assert int(
            local_counts.sum()
        ) == len(
            positions
        )

        stratum_positions[
            int(
                count
            )
        ] = positions

        stratum_local_counts[
            int(
                count
            )
        ] = local_counts

    reconstructed_counts = np.zeros(
        N_SUBWINDOWS_PER_PARENT,
        dtype=np.int64,
    )

    reconstructed_denominators = np.zeros(
        N_SUBWINDOWS_PER_PARENT,
        dtype=np.float64,
    )

    for count, local_counts in stratum_local_counts.items():
        p = (
            count
            / EXPECTED_PATHOGENS
        )

        weight = (
            p
            * (
                1.0
                - p
            )
        )

        reconstructed_counts += local_counts

        reconstructed_denominators += (
            local_counts
            * weight
        )

    observed_counts = parent_subwindow_rows[
        "n_unitigs"
    ].to_numpy(
        dtype=np.int64
    )

    observed_denominators = subwindow_denominators[
        global_codes
    ]

    assert np.array_equal(
        reconstructed_counts,
        observed_counts,
    )

    assert np.allclose(
        reconstructed_denominators,
        observed_denominators,
        atol=1e-10,
        rtol=1e-12,
    )

    parent_randomization_data[
        parent_name
    ] = {
        "parent_order": parent_order,
        "parent_unitig_indices": parent_unitig_indices,
        "global_codes": global_codes,
        "observed_counts": observed_counts,
        "observed_denominators": observed_denominators,
        "stratum_positions": stratum_positions,
        "stratum_local_counts": stratum_local_counts,
    }

def matched_random_local_code(
    parent_name,
    replicate_index,
):
    info = parent_randomization_data[
        parent_name
    ]

    rng = np.random.default_rng(
        RANDOM_SEED_BASE
        + (
            int(
                info[
                    "parent_order"
                ]
            )
            * 100_000
        )
        + int(
            replicate_index
        )
    )

    n_parent_unitigs = len(
        info[
            "parent_unitig_indices"
        ]
    )

    random_local_code = np.empty(
        n_parent_unitigs,
        dtype=np.int8,
    )

    for count, positions in info[
        "stratum_positions"
    ].items():
        local_counts = info[
            "stratum_local_counts"
        ][
            count
        ]

        labels = np.repeat(
            np.arange(
                N_SUBWINDOWS_PER_PARENT,
                dtype=np.int8,
            ),
            local_counts,
        )

        assert len(
            labels
        ) == len(
            positions
        )

        rng.shuffle(
            labels
        )

        random_local_code[
            positions
        ] = labels

    return random_local_code

smoke_start = time.time()
smoke_rows = []

for parent_name in PRIORITY_PARENTS:
    info = parent_randomization_data[
        parent_name
    ]

    random_local_code = matched_random_local_code(
        parent_name,
        0,
    )

    random_counts = np.bincount(
        random_local_code,
        minlength=N_SUBWINDOWS_PER_PARENT,
    ).astype(
        np.int64
    )

    assert np.array_equal(
        random_counts,
        info[
            "observed_counts"
        ],
    )

    for local_code in range(
        N_SUBWINDOWS_PER_PARENT
    ):
        columns = info[
            "parent_unitig_indices"
        ][
            random_local_code
            == local_code
        ]

        (
            random_numerator,
            random_denominator,
        ) = numerator_from_columns(
            columns
        )

        expected_denominator = float(
            info[
                "observed_denominators"
            ][
                local_code
            ]
        )

        assert abs(
            random_denominator
            - expected_denominator
        ) < 1e-10

        leave_out_K = (
            total_numerator
            - random_numerator
        ) / (
            total_denominator
            - expected_denominator
        )

        leave_out_K = (
            leave_out_K
            + leave_out_K.T
        ) / 2.0

        prepared = prepare_kernel(
            leave_out_K
        )

        fit = fit_null_reml_prepared(
            y,
            prepared,
        )

        random_fraction = float(
            fit[
                "variance_fraction"
            ]
        )

        random_drop = (
            baseline_variance_fraction
            - random_fraction
        )

        global_code = int(
            info[
                "global_codes"
            ][
                local_code
            ]
        )

        subwindow_name = str(
            subwindow_manifest.loc[
                subwindow_manifest[
                    "global_subwindow_code"
                ]
                == global_code,
                "subwindow_name",
            ].iloc[0]
        )

        smoke_rows.append(
            {
                "parent_group_name": parent_name,
                "local_subwindow_code": local_code,
                "subwindow_name": subwindow_name,
                "matched_unitigs": len(
                    columns
                ),
                "random_leave_one_subwindow_out_variance_fraction": random_fraction,
                "random_drop_from_full": random_drop,
            }
        )

smoke_results = pd.DataFrame(
    smoke_rows
)

smoke_elapsed_seconds = (
    time.time()
    - smoke_start
)

estimated_total_minutes = (
    smoke_elapsed_seconds
    * N_RANDOM_PARTITIONS_PER_PARENT
    / 60.0
)

smoke_results.to_csv(
    SMOKE_TEST_FILE,
    index=False,
)

print("Matched-random subwindow smoke test: PASS")
print(
    "One partition for all four parents elapsed:",
    f"{smoke_elapsed_seconds:.1f}",
    "seconds",
)
print(
    "Estimated 250-partition total runtime:",
    f"{estimated_total_minutes:.1f}",
    "minutes",
)
print(
    "The long benchmark is restartable after every completed "
    "parent-partition."
)

display(
    smoke_results
)

print(
    "\nCell 21.5 complete."
)

print(
    "Transition: Cell 21.6 will run or resume 250 matched-random "
    "partitions within each mapped parent."
)


In [ ]:
#@title Cell 21.6 - Run or resume matched-random subwindow benchmarks
# Purpose:
# Generate 250 matched-random partitions inside each of the four parent
# windows.
#
# Progress is saved after every completed parent-partition.
# If Colab disconnects, rerun Cells 21.1-21.6; completed work is skipped.

progress_columns = [
    "parent_group_name",
    "replicate",
    "local_subwindow_code",
    "subwindow_name",
    "random_leave_one_subwindow_out_variance_fraction",
    "random_drop_from_full",
]

if PROGRESS_FILE.exists():
    progress = pd.read_csv(
        PROGRESS_FILE
    )

else:
    progress = pd.DataFrame(
        columns=progress_columns
    )

completed_parent_replicates = set()

if len(
    progress
) > 0:
    completed_counts = (
        progress.groupby(
            [
                "parent_group_name",
                "replicate",
            ]
        )[
            "local_subwindow_code"
        ]
        .nunique()
    )

    completed_parent_replicates = set(
        (
            str(
                parent_name
            ),
            int(
                replicate
            ),
        )
        for (
            parent_name,
            replicate
        ), count in completed_counts.items()
        if count == N_SUBWINDOWS_PER_PARENT
    )

total_parent_partitions = (
    len(
        PRIORITY_PARENTS
    )
    * N_RANDOM_PARTITIONS_PER_PARENT
)

print(
    "Already completed parent-partitions:",
    len(
        completed_parent_replicates
    ),
    "/",
    total_parent_partitions,
)

benchmark_start = time.time()

for parent_name in PRIORITY_PARENTS:
    info = parent_randomization_data[
        parent_name
    ]

    for replicate_index in range(
        N_RANDOM_PARTITIONS_PER_PARENT
    ):
        key = (
            parent_name,
            replicate_index,
        )

        if key in completed_parent_replicates:
            continue

        replicate_start = time.time()

        random_local_code = matched_random_local_code(
            parent_name,
            replicate_index,
        )

        random_counts = np.bincount(
            random_local_code,
            minlength=N_SUBWINDOWS_PER_PARENT,
        ).astype(
            np.int64
        )

        assert np.array_equal(
            random_counts,
            info[
                "observed_counts"
            ],
        )

        replicate_rows = []

        for local_code in range(
            N_SUBWINDOWS_PER_PARENT
        ):
            columns = info[
                "parent_unitig_indices"
            ][
                random_local_code
                == local_code
            ]

            (
                random_numerator,
                random_denominator,
            ) = numerator_from_columns(
                columns
            )

            expected_denominator = float(
                info[
                    "observed_denominators"
                ][
                    local_code
                ]
            )

            assert abs(
                random_denominator
                - expected_denominator
            ) < 1e-10

            leave_out_K = (
                total_numerator
                - random_numerator
            ) / (
                total_denominator
                - expected_denominator
            )

            leave_out_K = (
                leave_out_K
                + leave_out_K.T
            ) / 2.0

            prepared = prepare_kernel(
                leave_out_K
            )

            fit = fit_null_reml_prepared(
                y,
                prepared,
            )

            random_fraction = float(
                fit[
                    "variance_fraction"
                ]
            )

            random_drop = (
                baseline_variance_fraction
                - random_fraction
            )

            global_code = int(
                info[
                    "global_codes"
                ][
                    local_code
                ]
            )

            subwindow_name = str(
                subwindow_manifest.loc[
                    subwindow_manifest[
                        "global_subwindow_code"
                    ]
                    == global_code,
                    "subwindow_name",
                ].iloc[0]
            )

            replicate_rows.append(
                {
                    "parent_group_name": parent_name,
                    "replicate": replicate_index,
                    "local_subwindow_code": local_code,
                    "subwindow_name": subwindow_name,
                    "random_leave_one_subwindow_out_variance_fraction": random_fraction,
                    "random_drop_from_full": random_drop,
                }
            )

        if len(
            progress
        ) > 0:
            keep_mask = ~(
                (
                    progress[
                        "parent_group_name"
                    ].astype(
                        str
                    )
                    == parent_name
                )
                & (
                    progress[
                        "replicate"
                    ].astype(
                        int
                    )
                    == replicate_index
                )
            )

            progress = progress.loc[
                keep_mask
            ].copy()

        progress = pd.concat(
            [
                progress,
                pd.DataFrame(
                    replicate_rows
                ),
            ],
            ignore_index=True,
        )

        progress = (
            progress
            .sort_values(
                [
                    "parent_group_name",
                    "replicate",
                    "local_subwindow_code",
                ]
            )
            .reset_index(
                drop=True
            )
        )

        progress.to_csv(
            PROGRESS_FILE,
            index=False,
            compression="gzip",
        )

        completed_parent_replicates.add(
            key
        )

        elapsed_replicate = (
            time.time()
            - replicate_start
        )

        print(
            parent_name,
            "- completed partition:",
            replicate_index + 1,
            "/",
            N_RANDOM_PARTITIONS_PER_PARENT,
            "- total parent-partitions:",
            len(
                completed_parent_replicates
            ),
            "/",
            total_parent_partitions,
            "- elapsed:",
            f"{elapsed_replicate:.1f}",
            "seconds",
        )

progress = pd.read_csv(
    PROGRESS_FILE
)

completed_counts = (
    progress.groupby(
        [
            "parent_group_name",
            "replicate",
        ]
    )[
        "local_subwindow_code"
    ]
    .nunique()
)

assert len(
    completed_counts
) == total_parent_partitions

assert (
    completed_counts
    == N_SUBWINDOWS_PER_PARENT
).all()

expected_progress_rows = (
    len(
        PRIORITY_PARENTS
    )
    * N_RANDOM_PARTITIONS_PER_PARENT
    * N_SUBWINDOWS_PER_PARENT
)

assert len(
    progress
) == expected_progress_rows

print(
    "\nMatched-random subwindow benchmark complete."
)

print(
    "Parent-partitions:",
    total_parent_partitions,
)

print(
    "Saved rows:",
    len(
        progress
    ),
)

print(
    "Current-session elapsed:",
    f"{(time.time() - benchmark_start) / 60.0:.1f}",
    "minutes",
)

print(
    "\nCell 21.6 complete."
)

print(
    "Transition: Cell 21.7 will compare each observed subwindow "
    "drop with its matched-random distribution."
)


In [ ]:
#@title Cell 21.7 - Compare observed subwindow drops with matched-random distributions
# Purpose:
# Compare each observed subwindow removal with matched random removals drawn
# from the same parent window.
#
# Empirical p:
#   (1 + random drops >= observed drop) / (1 + 250)
#
# Because this is hierarchical refinement of four already selected parents,
# BH correction is applied separately across the 10 subwindows within each
# parent. The resulting q-values are within-parent refinement q-values.

progress = pd.read_csv(
    PROGRESS_FILE
)

observed_results = pd.read_csv(
    OBSERVED_RESULTS
)

expected_progress_rows = (
    len(
        PRIORITY_PARENTS
    )
    * N_RANDOM_PARTITIONS_PER_PARENT
    * N_SUBWINDOWS_PER_PARENT
)

assert len(
    progress
) == expected_progress_rows

assert len(
    observed_results
) == EXPECTED_SUBWINDOWS

def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
            dtype=float,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    result = np.empty(
        n,
        dtype=float,
    )

    result[
        order
    ] = adjusted

    return result

result_rows = []

for _, observed_row in observed_results.iterrows():
    parent_name = str(
        observed_row[
            "parent_group_name"
        ]
    )

    local_code = int(
        observed_row[
            "local_subwindow_code"
        ]
    )

    subwindow_name = str(
        observed_row[
            "subwindow_name"
        ]
    )

    observed_drop = float(
        observed_row[
            "absolute_drop_after_removal"
        ]
    )

    random_drops = (
        progress.loc[
            (
                progress[
                    "parent_group_name"
                ].astype(
                    str
                )
                == parent_name
            )
            & (
                progress[
                    "local_subwindow_code"
                ].astype(
                    int
                )
                == local_code
            ),
            "random_drop_from_full",
        ]
        .to_numpy(
            dtype=float
        )
    )

    assert len(
        random_drops
    ) == N_RANDOM_PARTITIONS_PER_PARENT

    n_equal_or_greater = int(
        np.sum(
            random_drops
            >= observed_drop
        )
    )

    empirical_p = float(
        (
            1
            + n_equal_or_greater
        )
        / (
            N_RANDOM_PARTITIONS_PER_PARENT
            + 1
        )
    )

    percentile = float(
        100.0
        * np.mean(
            random_drops
            <= observed_drop
        )
    )

    result_rows.append(
        {
            "global_subwindow_code": int(
                observed_row[
                    "global_subwindow_code"
                ]
            ),
            "parent_group_name": parent_name,
            "local_subwindow_code": local_code,
            "subwindow_name": subwindow_name,
            "reference_start_0_based": int(
                observed_row[
                    "reference_start_0_based"
                ]
            ),
            "reference_end_0_based_exclusive": int(
                observed_row[
                    "reference_end_0_based_exclusive"
                ]
            ),
            "n_unitigs": int(
                observed_row[
                    "n_unitigs"
                ]
            ),
            "fraction_of_parent_kernel_denominator": float(
                observed_row[
                    "fraction_of_parent_kernel_denominator"
                ]
            ),
            "observed_leave_one_subwindow_out_variance_fraction": float(
                observed_row[
                    "leave_one_subwindow_out_variance_fraction"
                ]
            ),
            "observed_drop_after_removal": observed_drop,
            "matched_random_mean_drop": float(
                np.mean(
                    random_drops
                )
            ),
            "matched_random_median_drop": float(
                np.median(
                    random_drops
                )
            ),
            "matched_random_95th_percentile": float(
                np.quantile(
                    random_drops,
                    0.95,
                )
            ),
            "matched_random_99th_percentile": float(
                np.quantile(
                    random_drops,
                    0.99,
                )
            ),
            "observed_drop_percentile": percentile,
            "random_drops_equal_or_greater": n_equal_or_greater,
            "empirical_p_value": empirical_p,
        }
    )

final_results = pd.DataFrame(
    result_rows
)

final_results[
    "within_parent_BH_q_value"
] = np.nan

for parent_name in PRIORITY_PARENTS:
    mask = (
        final_results[
            "parent_group_name"
        ]
        == parent_name
    )

    assert int(
        mask.sum()
    ) == N_SUBWINDOWS_PER_PARENT

    final_results.loc[
        mask,
        "within_parent_BH_q_value",
    ] = benjamini_hochberg(
        final_results.loc[
            mask,
            "empirical_p_value",
        ].to_numpy(
            dtype=float
        )
    )

final_results[
    "larger_drop_than_matched_random_after_within_parent_BH"
] = (
    final_results[
        "within_parent_BH_q_value"
    ]
    < 0.05
)

final_results[
    "within_parent_priority_rank"
] = (
    final_results.groupby(
        "parent_group_name"
    )[
        "empirical_p_value"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(
        int
    )
)

final_results = (
    final_results
    .sort_values(
        [
            "parent_group_name",
            "empirical_p_value",
            "observed_drop_after_removal",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

final_results.to_csv(
    FINAL_RESULTS,
    index=False,
)

display(
    final_results[
        [
            "parent_group_name",
            "within_parent_priority_rank",
            "subwindow_name",
            "reference_start_0_based",
            "reference_end_0_based_exclusive",
            "n_unitigs",
            "observed_drop_after_removal",
            "matched_random_median_drop",
            "matched_random_95th_percentile",
            "matched_random_99th_percentile",
            "observed_drop_percentile",
            "empirical_p_value",
            "within_parent_BH_q_value",
            "larger_drop_than_matched_random_after_within_parent_BH",
        ]
    ]
)

print(
    "\nCell 21.7 complete."
)

print(
    "Transition: Cell 21.8 will perform final QC and identify mapped "
    "subwindows, if any, justified for further refinement."
)


In [ ]:
#@title Cell 21.8 - Final QC and mapped-subwindow stopping decision
# Purpose:
# Freeze the first fine-scale mapped-window refinement.
#
# A subwindow is carried forward only if its observed removal caused a larger
# loss than matched random removals after BH correction within its parent.
#
# This is a hierarchical exploratory refinement. Passing subwindows are
# priorities for further narrowing, not causal regions or causal variants.

assert SUBWINDOW_ASSIGNMENT.exists()
assert SUBWINDOW_COMPONENTS.exists()
assert SUBWINDOW_MANIFEST.exists()
assert OBSERVED_RESULTS.exists()
assert PROGRESS_FILE.exists()
assert FINAL_RESULTS.exists()

saved_results = pd.read_csv(
    FINAL_RESULTS
)

progress = pd.read_csv(
    PROGRESS_FILE
)

assert len(
    saved_results
) == EXPECTED_SUBWINDOWS

expected_progress_rows = (
    len(
        PRIORITY_PARENTS
    )
    * N_RANDOM_PARTITIONS_PER_PARENT
    * N_SUBWINDOWS_PER_PARENT
)

assert len(
    progress
) == expected_progress_rows

assert saved_results[
    "empirical_p_value"
].between(
    0,
    1,
).all()

assert saved_results[
    "within_parent_BH_q_value"
].between(
    0,
    1,
).all()

priority_mask = saved_results[
    "larger_drop_than_matched_random_after_within_parent_BH"
].astype(
    bool
)

priority_subwindows = (
    saved_results.loc[
        priority_mask,
        [
            "parent_group_name",
            "subwindow_name",
            "reference_start_0_based",
            "reference_end_0_based_exclusive",
            "observed_drop_after_removal",
            "empirical_p_value",
            "within_parent_BH_q_value",
        ],
    ]
    .copy()
)

parent_summary_rows = []

for parent_name in PRIORITY_PARENTS:
    n_priority = int(
        (
            priority_subwindows[
                "parent_group_name"
            ]
            == parent_name
        ).sum()
    )

    parent_summary_rows.append(
        {
            "parent_group_name": parent_name,
            "subwindows_tested": N_SUBWINDOWS_PER_PARENT,
            "subwindows_passing_within_parent_BH_q_lt_0_05": n_priority,
        }
    )

parent_summary = pd.DataFrame(
    parent_summary_rows
)

qc = pd.DataFrame(
    [
        {
            "pathogens": EXPECTED_PATHOGENS,
            "variable_unitigs": EXPECTED_UNITIGS,
            "mapped_parent_windows_refined": len(
                PRIORITY_PARENTS
            ),
            "subwindows_per_parent": N_SUBWINDOWS_PER_PARENT,
            "total_subwindows_tested": EXPECTED_SUBWINDOWS,
            "random_partitions_per_parent": N_RANDOM_PARTITIONS_PER_PARENT,
            "progress_rows": len(
                progress
            ),
            "exact_parent_kernel_reconstruction_pass": True,
            "matched_unitig_count_pass": True,
            "matched_presence_count_distribution_pass": True,
            "matched_kernel_denominator_pass": True,
            "subwindows_passing_within_parent_BH_q_lt_0_05": len(
                priority_subwindows
            ),
            "final_QC_pass": True,
        }
    ]
)

qc.to_csv(
    FINAL_QC,
    index=False,
)

completion_payload = {
    "status": "complete",
    "mapped_parent_windows_refined": PRIORITY_PARENTS,
    "subwindows_per_parent": N_SUBWINDOWS_PER_PARENT,
    "matched_random_partitions_per_parent": N_RANDOM_PARTITIONS_PER_PARENT,
    "subwindows_passing_within_parent_BH_q_lt_0_05": priority_subwindows[
        "subwindow_name"
    ].astype(
        str
    ).tolist(),
    "final_QC_pass": True,
}

COMPLETION_FILE.write_text(
    json.dumps(
        completion_payload,
        indent=2,
    ),
    encoding="utf-8",
)

print("Final QC: PASS")

print(
    "\nMapped-parent summary:"
)

display(
    parent_summary
)

print(
    "\nFinal mapped-subwindow results:"
)

display(
    saved_results[
        [
            "parent_group_name",
            "within_parent_priority_rank",
            "subwindow_name",
            "reference_start_0_based",
            "reference_end_0_based_exclusive",
            "observed_drop_after_removal",
            "matched_random_95th_percentile",
            "empirical_p_value",
            "within_parent_BH_q_value",
            "larger_drop_than_matched_random_after_within_parent_BH",
        ]
    ]
)

if len(
    priority_subwindows
) > 0:
    print(
        "\nStopping decision: the following mapped subwindows are "
        "supported for further refinement within their parent windows:"
    )

    for _, row in priority_subwindows.iterrows():
        print(
            "-",
            row[
                "subwindow_name"
            ],
            ":",
            int(
                row[
                    "reference_start_0_based"
                ]
            ),
            "to",
            int(
                row[
                    "reference_end_0_based_exclusive"
                ]
            ),
            "bp",
        )

    print(
        "\nThese are refinement priorities only. They are not yet "
        "causal regions or causal variant combinations."
    )

else:
    print(
        "\nStopping decision: no mapped subwindow caused a larger loss "
        "than matched random removals after within-parent BH correction."
    )

    print(
        "In that case, the mapped parent-window effects should be treated "
        "as distributed within those windows rather than localized further "
        "by this 10-way coordinate split."
    )

print(
    "\nNotebook 21 ends here. Review these mapped-subwindow results "
    "before any further coordinate refinement and before refining "
    "UNMAPPED or MULTIMAPPED."
)
